<a href="https://colab.research.google.com/github/brunamportoDS/Weather/blob/main/01_sba_data_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset (Colab path after upload)
df = pd.read_csv("SBAnational.csv", low_memory=False)

# Step 1: Basic shape and size
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

# Step 2: Column names and types
print("COLUMNS & TYPES")
print("-" * 50)
for col in df.columns:
    non_null = df[col].notna().sum()
    pct_missing = (1 - non_null / len(df)) * 100
    print(f"  {col:<20} {str(df[col].dtype):<10} {non_null:>8,} non-null  ({pct_missing:.1f}% missing)")

Shape: 899,164 rows × 27 columns
Memory: 930.2 MB

COLUMNS & TYPES
--------------------------------------------------
  LoanNr_ChkDgt        int64       899,164 non-null  (0.0% missing)
  Name                 object      899,150 non-null  (0.0% missing)
  City                 object      899,134 non-null  (0.0% missing)
  State                object      899,150 non-null  (0.0% missing)
  Zip                  int64       899,164 non-null  (0.0% missing)
  Bank                 object      897,605 non-null  (0.2% missing)
  BankState            object      897,598 non-null  (0.2% missing)
  NAICS                int64       899,164 non-null  (0.0% missing)
  ApprovalDate         object      899,164 non-null  (0.0% missing)
  ApprovalFY           object      899,164 non-null  (0.0% missing)
  Term                 int64       899,164 non-null  (0.0% missing)
  NoEmp                int64       899,164 non-null  (0.0% missing)
  NewExist             float64     899,028 non-null  (0.0% missing

In [ ]:
# Step 3: Check the target variable
print("TARGET VARIABLE: MIS_Status")
print(df['MIS_Status'].value_counts(dropna=False))
print()

# Step 4: Peek at currency columns (confirm $ and commas)
print("CURRENCY COLUMNS - Sample values:")
for col in ['DisbursementGross', 'BalanceGross', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv']:
    print(f"  {col}: {df[col].iloc[0]}")
print()

# Step 5: Check NAICS codes (we need 2-digit sector codes for Census join)
print("NAICS - Sample values and range:")
print(f"  Min: {df['NAICS'].min()}, Max: {df['NAICS'].max()}")
print(f"  Sample: {df['NAICS'].head(10).tolist()}")
print(f"  Unique 2-digit sectors: {df['NAICS'].astype(str).str[:2].nunique()}")
print()

# Step 6: Check date formats
print("DATE COLUMNS - Sample values:")
print(f"  ApprovalDate: {df['ApprovalDate'].iloc[0]}")
print(f"  DisbursementDate: {df['DisbursementDate'].iloc[0]}")
print()

# Step 7: Check State values (for Census join)
print(f"STATES: {df['State'].nunique()} unique values")
print(f"  Sample: {df['State'].value_counts().head(10).to_dict()}")

TARGET VARIABLE: MIS_Status
MIS_Status
P I F     739609
CHGOFF    157558
NaN         1997
Name: count, dtype: int64

CURRENCY COLUMNS - Sample values:
  DisbursementGross: $60,000.00 
  BalanceGross: $0.00 
  ChgOffPrinGr: $0.00 
  GrAppv: $60,000.00 
  SBA_Appv: $48,000.00 

NAICS - Sample values and range:
  Min: 0, Max: 928120
  Sample: [451120, 722410, 621210, 0, 0, 332721, 0, 811118, 721310, 0]
  Unique 2-digit sectors: 25

DATE COLUMNS - Sample values:
  ApprovalDate: 28-Feb-97
  DisbursementDate: 28-Feb-99

STATES: 51 unique values
  Sample: {'CA': 130619, 'TX': 70458, 'NY': 57693, 'FL': 41212, 'PA': 35170, 'OH': 32622, 'IL': 29669, 'MA': 25272, 'MN': 24373, 'NJ': 24035}


In [ ]:
# Step 8: How many NAICS = 0 (missing industry)?
naics_zero = (df['NAICS'] == 0).sum()
print(f"NAICS = 0 (missing): {naics_zero:,} ({naics_zero/len(df)*100:.1f}%)")
print()

# Step 9: Distribution of 2-digit NAICS sectors (excluding 0)
naics_2digit = df['NAICS'].astype(str).str[:2]
print("TOP 2-DIGIT NAICS SECTORS:")
print(naics_2digit[naics_2digit != '0'].value_counts().head(15))
print()

# Step 10: ApprovalFY — check the year range
print("APPROVAL YEAR RANGE:")
print(df['ApprovalFY'].value_counts().sort_index())
print()

# Step 11: Quick look at other categorical columns
print("NewExist values:", df['NewExist'].value_counts(dropna=False).to_dict())
print("RevLineCr values:", df['RevLineCr'].value_counts(dropna=False).to_dict())
print("LowDoc values:", df['LowDoc'].value_counts(dropna=False).to_dict())
print("UrbanRural values:", df['UrbanRural'].value_counts(dropna=False).to_dict())

NAICS = 0 (missing): 201,948 (22.5%)

TOP 2-DIGIT NAICS SECTORS:
NAICS
44    84737
81    72618
54    68170
72    67600
23    66646
62    55366
42    48743
45    42514
33    38284
56    32685
48    20310
32    17936
71    14640
53    13632
31    11809
Name: count, dtype: int64

APPROVAL YEAR RANGE:
ApprovalFY
1962         1
1965         1
1966         1
1967         2
1968         2
1969         4
1970         8
1971        20
1972        27
1973        52
1974        42
1975        30
1976        66
1976A       18
1977       137
1978       242
1979       352
1980       477
1981       630
1982       719
1983      1684
1984      2022
1985      1944
1986      2118
1987      2218
1988      1898
1989     13248
1990     14859
1991     15666
1992     20885
1993     23305
1994     31598
1995     45758
1996     40112
1997     37748
1998     36016
1999     37363
2000     37381
2001     37350
2002     44391
2003     58193
2004     68290
2005     77525
2006     76040
2007     71876
2008     39540


In [ ]:
# PROFILING SUMMARY
print("=" * 60)
print("SBA NATIONAL DATASET — PROFILING SUMMARY")
print("=" * 60)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Year range: 1962–2014 (recommend filtering to 1987+)")

total = len(df)
mis_null = df['MIS_Status'].isna().sum()
naics_zero = (df['NAICS'] == 0).sum()
default_rate = (df['MIS_Status'].str.strip() == 'CHGOFF').sum() / df['MIS_Status'].notna().sum() * 100

print(f"\nTarget variable (MIS_Status):")
print(f"  PIF (paid in full): {(df['MIS_Status'].str.strip() == 'P I F').sum():,}")
print(f"  CHGOFF (default):   {(df['MIS_Status'].str.strip() == 'CHGOFF').sum():,}")
print(f"  Missing:            {mis_null:,} ({mis_null/total*100:.1f}%)")
print(f"  Default rate:       {default_rate:.1f}%")

print(f"\nDATA QUALITY ISSUES TO ADDRESS:")
print(f"  1. NAICS = 0 (missing industry):     {naics_zero:,} rows ({naics_zero/total*100:.1f}%)")
print(f"  2. MIS_Status missing:               {mis_null:,} rows — drop these")
print(f"  3. MIS_Status has spaces:            'P I F' needs stripping")
print(f"  4. Currency columns as strings:       DisbursementGross, BalanceGross, ChgOffPrinGr, GrAppv, SBA_Appv")
print(f"  5. Date columns as strings:           ApprovalDate, DisbursementDate, ChgOffDate")
print(f"  6. RevLineCr dirty values:            {(~df['RevLineCr'].isin(['Y','N'])).sum():,} rows not Y/N")
print(f"  7. LowDoc dirty values:              {(~df['LowDoc'].isin(['Y','N'])).sum():,} rows not Y/N")
print(f"  8. NewExist = 0 (unknown):           {(df['NewExist'] == 0).sum():,} rows")
print(f"  9. ApprovalFY '1976A':               {(df['ApprovalFY'] == '1976A').sum()} rows")
print(f"  10. Pre-1987 sparse data:            Consider filtering to 1987+")

print(f"\nJOIN KEY READINESS:")
print(f"  NAICS 2-digit sectors: {df['NAICS'].astype(str).str[:2].nunique()} unique (excl. 0)")
print(f"  States: {df['State'].nunique()} unique — clean for Census join")
print(f"  Dates: need parsing for FRED join by year/month")

SBA NATIONAL DATASET — PROFILING SUMMARY

Shape: 899,164 rows × 27 columns
Memory: 930.2 MB
Year range: 1962–2014 (recommend filtering to 1987+)

Target variable (MIS_Status):
  PIF (paid in full): 739,609
  CHGOFF (default):   157,558
  Missing:            1,997 (0.2%)
  Default rate:       17.6%

DATA QUALITY ISSUES TO ADDRESS:
  1. NAICS = 0 (missing industry):     201,948 rows (22.5%)
  2. MIS_Status missing:               1,997 rows — drop these
  3. MIS_Status has spaces:            'P I F' needs stripping
  4. Currency columns as strings:       DisbursementGross, BalanceGross, ChgOffPrinGr, GrAppv, SBA_Appv
  5. Date columns as strings:           ApprovalDate, DisbursementDate, ChgOffDate
  6. RevLineCr dirty values:            277,479 rows not Y/N
  7. LowDoc dirty values:              6,007 rows not Y/N
  8. NewExist = 0 (unknown):           1,034 rows
  9. ApprovalFY '1976A':               18 rows
  10. Pre-1987 sparse data:            Consider filtering to 1987+

JOIN KEY RE